# LLM-as-Judge (Verification MCP)

Implement and test the LLM-as-Judge pattern with a structured evaluation rubric.

## Learning Objectives
- Understand rubric-based LLM evaluation
- Build structured judgment prompts
- Calibrate pass/fail thresholds
- See how the judge drives the harness iteration loop

In [ ]:
import os
import sys
import httpx
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv
env_path = os.path.join('..', '.env')
load_dotenv(env_path, override=True)

_VERIFY_SSL = os.getenv("VERIFY_SSL", "true").lower() not in ("false", "0", "no")
_http_client = None if _VERIFY_SSL else httpx.Client(verify=False, timeout=httpx.Timeout(300.0))

from agents.orchestrator.layers.verification import llm_as_judge
print("LLM-as-Judge loaded")

## 1. The Rubric

| Criterion | Score | Description |
|-----------|-------|-------------|
| Relevance | 0-2 | Does it answer the question? |
| Depth | 0-2 | Is it thorough and detailed? |
| Evidence | 0-2 | Are claims supported with citations? |
| Clarity | 0-2 | Is it well-organized and readable? |
| Completeness | 0-2 | Does it cover all aspects? |

**Pass threshold**: Total >= 7/10

In [ ]:
# Test with a high-quality draft
query = "Explain the architecture of a RAG system"

good_draft = """
# RAG System Architecture

## Executive Summary
Retrieval-Augmented Generation (RAG) combines information retrieval with language model
generation to produce grounded, factual responses [Source 1].

## Core Components

### 1. Document Ingestion Pipeline
Documents are parsed, chunked, and embedded into vector representations [Source 1].
Key steps include: parsing (Docling), chunking (semantic boundaries), embedding
(sentence transformers), and storage (pgvector) [Source 2].

### 2. Retrieval Engine
Semantic similarity search finds relevant chunks [Source 2]:
- Query embedding generation
- Approximate nearest neighbor search (HNSW)
- Re-ranking for relevance

### 3. Generation Component
The LLM generates responses grounded in retrieved context [Source 1].

## Conclusion
RAG architecture enables factual, cited responses by combining retrieval with generation.
"""

result = llm_as_judge(good_draft, query, iteration=1)
print("High-quality draft evaluation:")
print(f"  Relevance: {result.get('relevance', 'N/A')}/2")
print(f"  Depth: {result.get('depth', 'N/A')}/2")
print(f"  Evidence: {result.get('evidence', 'N/A')}/2")
print(f"  Clarity: {result.get('clarity', 'N/A')}/2")
print(f"  Completeness: {result.get('completeness', 'N/A')}/2")
print(f"  Total: {result.get('total', 'N/A')}/10")
print(f"  Verdict: {result.get('verdict', 'N/A')}")
print(f"  Reasoning: {result.get('reasoning', 'N/A')}")

In [ ]:
# Test with a low-quality draft
bad_draft = "RAG is good. It uses vectors. The end."

result = llm_as_judge(bad_draft, query, iteration=1)
print("Low-quality draft evaluation:")
print(f"  Total: {result.get('total', 'N/A')}/10")
print(f"  Verdict: {result.get('verdict', 'N/A')}")
print(f"  Reasoning: {result.get('reasoning', 'N/A')}")
print(f"  Improvements: {result.get('improvements', [])}")

## 2. How the Judge Drives Iteration

When the judge returns `verdict: 'fail'`, the improvements list feeds into the next iteration's failure hints:

```python
# In the observe node:
if not passed:
    failure_hints = improvements joined as bullet points
    → feeds into next plan_node context
    → planner generates targeted research steps
```

In [ ]:
# Demonstrate how improvements become failure hints
improvements = result.get('improvements', [])
if improvements:
    failure_hints = "\n".join(f"- {imp}" for imp in improvements)
    print("Failure hints for next iteration:")
    print(failure_hints)
else:
    print("No improvements needed (draft passed)")

## Summary

LLM-as-Judge:
- Provides **structured evaluation** using a 5-criterion rubric
- Returns **actionable improvements** that guide the next iteration
- Uses a **pass/fail verdict** based on total score threshold
- Enables **calibration** by adjusting the threshold